Data Dokumen

In [9]:
import pandas as pd

kalimat_satu  = "Saya minum kopi hangat setiap pagi sebelum berangkat kuliah"
kalimat_dua   = "Kopi membantu saya tetap semangat saat belajar di rumah"
kalimat_tiga  = "Hari ini saya belajar coding untuk ujian besok"

daftar_dokumen = [kalimat_satu, kalimat_dua, kalimat_tiga]

Tokenisasi & Vocabulary

In [10]:
token_unik = []

for kata in kalimat_satu.lower().split():
    if kata not in token_unik:
        token_unik.append(kata)

for kata in kalimat_dua.lower().split():
    if kata not in token_unik:
        token_unik.append(kata)

for kata in kalimat_tiga.lower().split():
    if kata not in token_unik:
        token_unik.append(kata)

In [11]:
token_unik

['saya',
 'minum',
 'kopi',
 'hangat',
 'setiap',
 'pagi',
 'sebelum',
 'berangkat',
 'kuliah',
 'membantu',
 'tetap',
 'semangat',
 'saat',
 'belajar',
 'di',
 'rumah',
 'hari',
 'ini',
 'coding',
 'untuk',
 'ujian',
 'besok']

Membuat Vektor 0/1 untuk Setiap Dokumen

In [12]:
vektor_doc1 = []
vektor_doc2 = []
vektor_doc3 = []

for kata in token_unik:
    vektor_doc1.append(1 if kata in kalimat_satu.lower().split() else 0)

for kata in token_unik:
    vektor_doc2.append(1 if kata in kalimat_dua.lower().split() else 0)

for kata in token_unik:
    vektor_doc3.append(1 if kata in kalimat_tiga.lower().split() else 0)

print("vektor doc1:", vektor_doc1)
print("vektor doc2:", vektor_doc2)
print("vektor doc3:", vektor_doc3)


vektor doc1: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
vektor doc2: [1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
vektor doc3: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1]


Term–Document Incidence Matrix

In [13]:
matriks_insidensi = pd.DataFrame({
    "term": token_unik,
    "doc1": vektor_doc1,
    "doc2": vektor_doc2,
    "doc3": vektor_doc3
})

matriks_insidensi

,term,doc1,doc2,doc3
0,saya,1,1,1
1,minum,1,0,0
2,kopi,1,1,0
3,hangat,1,0,0
4,setiap,1,0,0
5,pagi,1,0,0
6,sebelum,1,0,0
7,berangkat,1,0,0
8,kuliah,1,0,0
9,membantu,0,1,0


Fungsi Term Vector dan Operasi Boolean

In [14]:
def ambil_vektor_term(term: str):
    """Mengambil vektor term dari matriks insidensi."""
    term = term.lower()
    baris = matriks_insidensi[matriks_insidensi["term"] == term]
    if len(baris) == 0:
        return [0, 0, 0]  # jika term tidak ditemukan
    return baris[["doc1", "doc2", "doc3"]].values.tolist()[0]

def operasi_NOT(vektor):
    """Membalik 0<->1 pada vektor."""
    return [1 - x for x in vektor]

def operasi_AND(vektor_a, vektor_b):
    """AND per posisi vektor."""
    return [vektor_a[i] & vektor_b[i] for i in range(len(vektor_a))]

def operasi_OR(vektor_a, vektor_b):
    """OR per posisi vektor."""
    return [vektor_a[i] | vektor_b[i] for i in range(len(vektor_a))]

def dokumen_cocok_dari_vektor(vektor_hasil):
    """Mengubah vektor hasil menjadi daftar nomor dokumen yang bernilai 1."""
    hasil = []
    for i, nilai in enumerate(vektor_hasil):
        if nilai == 1:
            hasil.append(i + 1)
    return hasil

Menjalankan Query dari User

In [15]:
print("=== INPUT QUERY BOOLEAN ===")
print("Format yang didukung:")
print("1) kata1 AND kata2")
print("2) kata1 OR kata2")
print("3) kata1 AND NOT kata2")
print("4) kata1 OR NOT kata2")
print("Contoh: kopi AND NOT coding\n")

query = input("Masukkan query: ").strip()
bagian = query.split()
bagian = [b.lower() for b in bagian]

if len(bagian) not in (3, 4):
    raise ValueError("Query tidak valid. Contoh: kopi AND NOT coding")

kata_kiri = bagian[0]
operator = bagian[1]

if operator not in ("and", "or"):
    raise ValueError("Operator harus AND atau OR")

vektor_kiri = ambil_vektor_term(kata_kiri)

if len(bagian) == 3:
    kata_kanan = bagian[2]
    vektor_kanan = ambil_vektor_term(kata_kanan)
else:
    if bagian[2] != "not":
        raise ValueError("Format NOT harus: kata1 AND NOT kata2")
    kata_kanan = bagian[3]
    vektor_kanan = operasi_NOT(ambil_vektor_term(kata_kanan))

if operator == "and":
    vektor_hasil = operasi_AND(vektor_kiri, vektor_kanan)
else:
    vektor_hasil = operasi_OR(vektor_kiri, vektor_kanan)

print("\n=== PROSES ===")
print("Query:", query.upper())
print(f"Vektor '{kata_kiri}':", vektor_kiri)

if len(bagian) == 4:
    print(f"Vektor '{kata_kanan}':", ambil_vektor_term(kata_kanan))
    print(f"Vektor 'NOT {kata_kanan}':", vektor_kanan)
else:
    print(f"Vektor '{kata_kanan}':", vektor_kanan)

print("Vektor hasil:", vektor_hasil)

daftar_hasil = dokumen_cocok_dari_vektor(vektor_hasil)
print("Dokumen yang cocok:", [f"doc{i}" for i in daftar_hasil])

for i in daftar_hasil:
    print(f"- doc{i}: {daftar_dokumen[i-1]}")

=== INPUT QUERY BOOLEAN ===
Format yang didukung:
1) kata1 AND kata2
2) kata1 OR kata2
3) kata1 AND NOT kata2
4) kata1 OR NOT kata2
Contoh: kopi AND NOT coding


=== PROSES ===
Query: KOPI AND SAYA
Vektor 'kopi': [1, 1, 0]
Vektor 'saya': [1, 1, 1]
Vektor hasil: [1, 1, 0]
Dokumen yang cocok: ['doc1', 'doc2']
- doc1: Saya minum kopi hangat setiap pagi sebelum berangkat kuliah
- doc2: Kopi membantu saya tetap semangat saat belajar di rumah


Contoh Query yang Bisa Dicoba

In [16]:
print("=== INPUT QUERY BOOLEAN ===")
print("Format yang didukung:")
print("1) kata1 AND kata2")
print("2) kata1 OR kata2")
print("3) kata1 AND NOT kata2")
print("4) kata1 OR NOT kata2")
print("Contoh: kopi AND NOT coding\n")

query = input("Masukkan query: ").strip()
bagian = query.split()
bagian = [b.lower() for b in bagian]

if len(bagian) not in (3, 4):
    raise ValueError("Query tidak valid. Contoh: kopi AND NOT coding")

kata_kiri = bagian[0]
operator = bagian[1]

if operator not in ("and", "or"):
    raise ValueError("Operator harus AND atau OR")

vektor_kiri = ambil_vektor_term(kata_kiri)

if len(bagian) == 3:
    kata_kanan = bagian[2]
    vektor_kanan = ambil_vektor_term(kata_kanan)
else:
    if bagian[2] != "not":
        raise ValueError("Format NOT harus: kata1 AND NOT kata2")
    kata_kanan = bagian[3]
    vektor_kanan = operasi_NOT(ambil_vektor_term(kata_kanan))

if operator == "and":
    vektor_hasil = operasi_AND(vektor_kiri, vektor_kanan)
else:
    vektor_hasil = operasi_OR(vektor_kiri, vektor_kanan)

print("\n=== PROSES ===")
print("Query:", query.upper())
print(f"Vektor '{kata_kiri}':", vektor_kiri)

if len(bagian) == 4:
    print(f"Vektor '{kata_kanan}':", ambil_vektor_term(kata_kanan))
    print(f"Vektor 'NOT {kata_kanan}':", vektor_kanan)
else:
    print(f"Vektor '{kata_kanan}':", vektor_kanan)

print("Vektor hasil:", vektor_hasil)

daftar_hasil = dokumen_cocok_dari_vektor(vektor_hasil)
print("Dokumen yang cocok:", [f"doc{i}" for i in daftar_hasil])

for i in daftar_hasil:
    print(f"- doc{i}: {daftar_dokumen[i-1]}")


=== INPUT QUERY BOOLEAN ===
Format yang didukung:
1) kata1 AND kata2
2) kata1 OR kata2
3) kata1 AND NOT kata2
4) kata1 OR NOT kata2
Contoh: kopi AND NOT coding


=== PROSES ===
Query: KOPI AND NOT CODING
Vektor 'kopi': [1, 1, 0]
Vektor 'coding': [0, 0, 1]
Vektor 'NOT coding': [1, 1, 0]
Vektor hasil: [1, 1, 0]
Dokumen yang cocok: ['doc1', 'doc2']
- doc1: Saya minum kopi hangat setiap pagi sebelum berangkat kuliah
- doc2: Kopi membantu saya tetap semangat saat belajar di rumah
